In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 42
N_PERTURB = 5000  # same scale as LIME/SHAP for fairness

DATA_DIR = Path("data_processed")
MLP_DIR = Path("artifacts/mlp")
LIME_DIR = Path("artifacts/lime")
SHAP_DIR = Path("artifacts/shap")
OUT_DIR = Path("artifacts/leaf")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Data
X_train = np.load(DATA_DIR / "X_train.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

meta = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
feature_names = meta["feature_names"]

# Model
model = joblib.load(MLP_DIR / "model.joblib")

# Indices (same as LIME/SHAP)
chosen = json.loads((LIME_DIR / "chosen_test_indices.json").read_text(encoding="utf-8"))
test_indices = [int(i) for i in chosen["chosen_test_indices"]]

len(test_indices)

In [ ]:
# LIME
lime_summary = json.loads((LIME_DIR / "lime_summary.json").read_text(encoding="utf-8"))

# SHAP
shap_values = np.load(SHAP_DIR / "shap_values_class1.npy")
expected_value = json.loads((SHAP_DIR / "expected_value.json").read_text(encoding="utf-8"))["expected_value"]

# Map LIME results by index for convenience
lime_map = {rec["test_index"]: rec for rec in lime_summary["results"]}

In [ ]:
def generate_perturbations(x, X_train, n_samples, seed):
    rng = np.random.default_rng(seed)
    
    # Sample with replacement from training distribution
    idx = rng.choice(np.arange(X_train.shape[0]), size=n_samples, replace=True)
    Z = X_train[idx].copy()  # shape (n_samples, n_features)
    
    # Mask 50% of each feature in each sample, replace with x
    mask = rng.random(Z.shape) < 0.5  # shape (n_samples, n_features)
    
    # Broadcasting: replace Z where mask is True with corresponding x values
    Z[mask] = np.broadcast_to(x, Z.shape)[mask]
    
    return Z

In [ ]:
def lime_linear_model(weights, feature_names, x):
    # weights: list of {"feature": "...", "weight": ...}
    
    coef = np.zeros(len(feature_names), dtype=float)
    
    for w in weights:
        name = w["feature"]
        weight = w["weight"]
        
        # LIME uses conditions like "feature <= value"
        # We approximate by matching feature name prefix
        for i, fname in enumerate(feature_names):
            if fname in name:
                coef[i] = weight
                break
    
    intercept = 0.0  # LIME doesn't give explicit intercept
    
    def g(X):
        return X @ coef + intercept
    
    return g

In [ ]:
def shap_linear_model(shap_values, expected_value, x):
    phi = shap_values  # (n_features,)
    
    def g(X):
        return expected_value + np.sum(phi * (X - x), axis=1)
    
    return g

In [ ]:
def f_model(X):
    return model.predict_proba(X)[:, 1]  # class 1

In [ ]:
results = []

for i, idx in enumerate(test_indices):
    x = X_test[idx]
    
    # Generate neighborhood
    Z = generate_perturbations(x, X_train, N_PERTURB, SEED + idx)
    
    f_vals = f_model(Z)
    
    # --- LIME ---
    lime_rec = lime_map[idx]
    lime_g = lime_linear_model(lime_rec["weights"], feature_names, x)
    g_lime = lime_g(Z)
    
    mse_lime = mean_squared_error(f_vals, g_lime)
    fidelity_lime = 1.0 - mse_lime
    
    # --- SHAP ---
    shap_phi = shap_values[i]
    shap_g = shap_linear_model(shap_phi, expected_value, x)
    g_shap = shap_g(Z)
    
    mse_shap = mean_squared_error(f_vals, g_shap)
    fidelity_shap = 1.0 - mse_shap
    
    results.append({
        "test_index": int(idx),
        "lime_fidelity": float(fidelity_lime),
        "shap_fidelity": float(fidelity_shap),
        "lime_mse": float(mse_lime),
        "shap_mse": float(mse_shap),
    })

len(results)

In [ ]:
summary = {
    "seed": SEED,
    "n_perturb": N_PERTURB,
    "n_examples": len(results),
    "results": results,
    "avg_lime_fidelity": float(np.mean([r["lime_fidelity"] for r in results])),
    "avg_shap_fidelity": float(np.mean([r["shap_fidelity"] for r in results])),
}

(OUT_DIR / "leaf_fidelity.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

summary